# 🎨 Colab LoRA Studio - Master All-in-One
### Huấn Luyện LoRA Đa Kiến Trúc (Flux.1, Flux-Kontext, Krea2-Raw, SDXL, Pony v6, SD 3.5, SD 1.5)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nguyenducvuongg/LorasTrainningColab/blob/master/Colab_LoRA_Studio.ipynb)

---
* **Tự Động Nhận Diện GPU (T4, L4, A100)** và tối ưu hóa VRAM/RAM thích ứng.
* **Lưu trữ 100% Trực Tiếp Vào Google Drive**: Tải Base Models 1 lần dùng mãi mãi, tự động sao lưu Checkpoint chống mất kết nối.
* **Hỗ trợ Dataset Chuyên Biệt**: Face (khuôn mặt), Character (cả khuôn mặt, body, trang phục), Style, Skin Texture/Detailer, Control-LoRA, hoặc thư mục tùy chỉnh không giới hạn file.
* **Auto-Captioning Đa Nguồn**: Google Gemini 1.5/2.0 API, DeepSeek API, SmilingWolf WD14 Tagger v3, JoyCaption/Florence-2.
* **Chạy 1-Click (Run All Ready)**: Tích hợp bảng hướng dẫn chi tiết các thông số (Batch Size, Optimizer, Epochs, Steps, Learning Rate 1e-4).

In [ ]:
#@title 🚀 Cell 1: Khởi Tạo Môi Trường & Smart Google Drive Setup { vertical-output: true }
#@markdown Nhấn nút Run để kết nối Google Drive, khởi tạo thư mục chuẩn hóa và cài đặt môi trường.

import os
import sys

REPO_URL = "https://github.com/nguyenducvuongg/LorasTrainningColab.git" #@param {type:"string"}
WORKSPACE_DIR = "/content/LorasTrainningColab"

# 1. Kết Nối Google Drive Ngay Lập Tức (Hiển thị popup xác nhận tức thì)
if "google.colab" in sys.modules or os.path.exists("/content"):
    if not os.path.exists("/content/drive/MyDrive"):
        print("🚀 Đang yêu cầu cấp quyền Google Drive...")
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)

# 2. Clone / Update Repository từ GitHub
if not os.path.exists(WORKSPACE_DIR):
    print(f"📦 Đang tải kho mã nguồn từ {REPO_URL}...")
    !git clone {REPO_URL} {WORKSPACE_DIR}
else:
    print(f"🔄 Đang đồng bộ cập nhật mới nhất tại {WORKSPACE_DIR}...")
    !git -C {WORKSPACE_DIR} pull || true

%cd {WORKSPACE_DIR}
if "src" not in sys.path:
    sys.path.insert(0, os.path.join(WORKSPACE_DIR, "src"))

# 3. Khởi tạo cây thư mục chuẩn hóa trên Google Drive (Không ghi đè dữ liệu cũ)
from lora_colab.storage.gdrive_manager import GDriveWorkspaceManager
DRIVE_PATHS = GDriveWorkspaceManager.init_workspace()

# 4. Tự Động Tối Ưu Môi Trường Động (Chỉ quét & bổ sung gói lõi, hoàn tất sau vài giây)
from lora_colab.core.environment import AutoEnvironmentManager
AutoEnvironmentManager.optimize_and_install_dependencies(install_backends=False)

print("\n✅ Cell 1: Môi trường & Google Drive đã sẵn sàng 100%!")

In [ ]:
#@title ⚡ Cell 2: Nhận Diện Phần Cứng & Tự Động Tối Ưu (Auto Hardware Profile) { vertical-output: true }
#@markdown Quét GPU hiện tại của Google Colab (T4, L4, A100) và gán thông số tối ưu VRAM/RAM.

TARGET_FAMILY = "flux" #@param ["flux", "flux-kontext", "krea", "sdxl", "pony", "sd35", "sd15", "qwen"]

from lora_colab.core.hardware import HardwareProfiler
CURRENT_PROFILE = HardwareProfiler.detect_and_profile(TARGET_FAMILY)
HardwareProfiler.display_profile(CURRENT_PROFILE)
print("\n✅ Cell 2: Đã nhận diện và thiết lập VRAM Optimizer Profile!")

In [ ]:
#@title 📥 Cell 3: Smart Model Downloader (Lưu 100% Trực Tiếp Vào Drive) { vertical-output: true }
#@markdown Chọn mô hình nền tảng cần tải. Hệ thống sẽ quét Google Drive: nếu đã có sẽ bỏ qua, nếu thiếu sẽ tự động tải bù trực tiếp vào Drive.

SELECTED_MODEL = "flux-dev" #@param ["flux-dev", "flux-schnell", "flux-kontext", "krea2-raw", "z-image-kolors", "qwen-image", "sdxl-base", "pony-v6", "illustrious-xl", "sd35-medium", "sd15-base"]
HF_TOKEN = "" #@param {type:"string"}

from lora_colab.storage.model_downloader import SmartModelDownloader
from lora_colab.storage.gdrive_manager import GDriveWorkspaceManager

workspace_root = DRIVE_PATHS.get("root", GDriveWorkspaceManager.DEFAULT_DRIVE_ROOT)
model_file_path = SmartModelDownloader.ensure_model_ready(
    model_key=SELECTED_MODEL,
    workspace_root=workspace_root,
    hf_token=HF_TOKEN.strip() if HF_TOKEN.strip() else None
)

print(f"\n✅ Model sẵn sàng tại Google Drive: {model_file_path}")

In [ ]:
#@title 🖼️ Cell 4: Chuẩn Bị Dữ Liệu & Auto-Captioning Pipeline { vertical-output: true }
#@markdown Chọn phân loại tập dữ liệu hoặc nhập đường dẫn thư mục tùy chỉnh. Hệ thống không giới hạn số lượng và loại file trainning!

DATASET_CATEGORY = "01_face (Khuon mat: can mat, bieu cam, goc nghieng)" #@param ["01_face (Khuon mat: can mat, bieu cam, goc nghieng)", "02_character (Nhan vat tong the: Mat, Nua nguoi, Toan than, Body, Trang phuc)", "03_style (Phong cach nghe thuat, net ve, aesthetic)", "04_skin_enhancement (Tai tao chi tiet da, lo chan long, upscale)", "05_control (Control-LoRA / Cap anh dieu kien)", "06_custom_path (Thu muc tuy chinh bat ky tren Drive)"]
CUSTOM_DATASET_PATH = "" #@param {type:"string"}
FILENAME_PREFIX = "char" #@param {type:"string"}
ENABLE_FILE_RENAMING = True #@param {type:"boolean"}
SKIP_CAPTIONING = False #@param {type:"boolean"}
SCAN_EXISTING_CAPTIONS = True #@param {type:"boolean"}
SKIP_EXISTING_CAPTIONS = True #@param {type:"boolean"}
CAPTION_ENGINE = "gemini (Google Cloud Vision - 0% VRAM)" #@param ["gemini (Google Cloud Vision - 0% VRAM)", "florence2 (Florence-2 Large Local VLM)", "joycaption (JoyCaption Alpha Two LLM)", "deepseek (Cloud API OpenAI format)", "wd14 (Anime / Danbooru tags - Local)", "skip_captioning (Bo qua tao caption - Da co san file .txt)"]
TRIGGER_WORD = "sks person" #@param {type:"string"}
API_KEY = "" #@param {type:"string"}
TASK_TYPE = "character" #@param ["character", "style", "skin_enhancement", "general"]

import os
from lora_colab.dataset.normalizer import DatasetNormalizer
from lora_colab.dataset.cleaner import CaptionCleaner
from lora_colab.dataset.captioning.gemini_api import GeminiVisionCaptioner
from lora_colab.dataset.captioning.deepseek_api import DeepSeekVisionCaptioner
from lora_colab.dataset.captioning.wd14 import WD14Tagger
from lora_colab.dataset.captioning.florence2 import Florence2Captioner
from lora_colab.dataset.captioning.joycaption import JoyCaptioner

# Xác định thư mục dữ liệu mục tiêu
if "06_custom_path" in DATASET_CATEGORY and CUSTOM_DATASET_PATH.strip():
    target_dataset_dir = CUSTOM_DATASET_PATH.strip()
elif "01_face" in DATASET_CATEGORY:
    target_dataset_dir = os.path.join(DRIVE_PATHS["root"], "datasets", "01_face")
elif "02_character" in DATASET_CATEGORY:
    target_dataset_dir = os.path.join(DRIVE_PATHS["root"], "datasets", "02_character")
elif "03_style" in DATASET_CATEGORY:
    target_dataset_dir = os.path.join(DRIVE_PATHS["root"], "datasets", "03_style")
elif "04_skin_enhancement" in DATASET_CATEGORY:
    target_dataset_dir = os.path.join(DRIVE_PATHS["root"], "datasets", "04_skin_enhancement")
elif "05_control" in DATASET_CATEGORY:
    target_dataset_dir = os.path.join(DRIVE_PATHS["root"], "datasets", "05_control")
else:
    target_dataset_dir = os.path.join(DRIVE_PATHS["root"], "datasets", "02_character")

# Tự động phát hiện thư mục ảnh (kể cả khi để trong thư mục con như datasets/02_character/Mai_girl)
target_dataset_dir = DatasetNormalizer.resolve_dataset_dir(target_dataset_dir)
os.makedirs(target_dataset_dir, exist_ok=True)

# 1. Chuẩn hoá ảnh sang RGB
norm_result = DatasetNormalizer.normalize_folder(target_dataset_dir, prefix=FILENAME_PREFIX, enable_renaming=ENABLE_FILE_RENAMING)

# 2. Xử lý bước tạo Caption hoặc Bỏ qua theo tùy chọn
should_skip_caption = SKIP_CAPTIONING or ("skip_captioning" in CAPTION_ENGINE.lower())

if should_skip_caption:
    if SCAN_EXISTING_CAPTIONS:
        all_files = os.listdir(target_dataset_dir)
        valid_exts = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
        img_files = [f for f in all_files if os.path.splitext(f)[-1].lower() in valid_exts]
        txt_files = [f for f in all_files if f.endswith(".txt")]
        print(f"\n⚡ Bỏ qua bước tạo caption theo tùy chọn:")
        print(f"  • Tổng số ảnh hiện có: {len(img_files)}")
        print(f"  • Số file caption (.txt) hiện có: {len(txt_files)}")
        if len(txt_files) < len(img_files):
            print(f"  ℹ️ Lưu ý: Có {len(img_files) - len(txt_files)} ảnh chưa có file .txt riêng. Trainer sẽ tự động gán trigger word '{TRIGGER_WORD}'.")
        else:
            print(f"  ✓ Đã có đủ {len(txt_files)} file caption .txt tương ứng cho toàn bộ ảnh!")
    else:
        print(f"\n⚡ Đã bỏ qua toàn bộ bước quét và tạo caption! Chuyển thẳng sang Cell 5.")
else:
    caption_cache = os.path.join(DRIVE_PATHS["root"], "models", "captioners")
    os.makedirs(caption_cache, exist_ok=True)
    
    if "gemini" in CAPTION_ENGINE:
        captioner = GeminiVisionCaptioner(api_key=API_KEY.strip() if API_KEY.strip() else None, task_type=TASK_TYPE)
    elif "deepseek" in CAPTION_ENGINE:
        captioner = DeepSeekVisionCaptioner(api_key=API_KEY.strip() if API_KEY.strip() else None, task_type=TASK_TYPE)
    elif "florence2" in CAPTION_ENGINE:
        captioner = Florence2Captioner(task_mode=TASK_TYPE, cache_dir=caption_cache)
    elif "joycaption" in CAPTION_ENGINE:
        captioner = JoyCaptioner(task_mode=TASK_TYPE, cache_dir=caption_cache)
    elif "wd14" in CAPTION_ENGINE:
        captioner = WD14Tagger()
    else:
        captioner = Florence2Captioner(task_mode=TASK_TYPE, cache_dir=caption_cache)
        
    captioner.caption_directory(target_dataset_dir, trigger_word=TRIGGER_WORD, skip_existing=SKIP_EXISTING_CAPTIONS)

print(f"\n✅ Dữ liệu tại '{target_dataset_dir}' đã sẵn sàng 100% để train!")

In [ ]:
#@title ⚙️ Cell 5: Cấu Hình Huấn Luyện (Training Configuration) { vertical-output: true }
#@markdown ### 📖 HƯỚNG DẪN THIẾT LẬP THÔNG SỐ:
#@markdown * **Training Engine**: Tự động chọn engine tối ưu nhất (`AI-Toolkit` cho Flux/Krea, `sd-scripts` cho SDXL/Pony, `musubi-tuner` cho Wan/Qwen).
#@markdown * **Batch Size**: Số ảnh xử lý cùng lúc (T4: `1`, L4: `2-4`, A100: `4-8`).
#@markdown * **Optimizer**: `Prodigy` (Tự động chỉnh LR tối ưu nhất - Bắt buộc đặt LR=`1.0`), `AdamW8bit` (Tiết kiệm VRAM, LR=`1e-4`), `AdamW` (A100 tốc độ cao).
#@markdown * **Epochs & Steps**: Số chu kỳ học qua dataset (Khuôn mặt / Nhân vật: `10 - 15`, Phong cách: `15 - 20`). Max Steps `0` = Tự tính theo Epochs.
#@markdown * **Network Rank & Alpha**: Dung lượng LoRA (Rank `32` + Alpha `16` hoặc Rank `64` + Alpha `32` cho độ nét tối đa).
#@markdown * **Learning Rate**: `1e-4` (0.0001 - Mặc định AdamW), `5e-5` (0.00005 - Tinh chỉnh nhẹ), `1.0` (Dành riêng cho Prodigy).

TRAINING_ENGINE = "auto (Tu Dong Chon Engine Toi Uu Theo Model)" #@param ["auto (Tu Dong Chon Engine Toi Uu Theo Model)", "ai-toolkit (Ostris Flux/Krea)", "sd-scripts (Kohya_ss SDXL/Pony/SD1.5)", "musubi-tuner (Wan/Qwen)", "diffusers"]
OUTPUT_LORA_NAME = "my_custom_lora" #@param {type:"string"}
BATCH_SIZE = 2 #@param [1, 2, 4, 8]
OPTIMIZER_CHOICE = "Prodigy" #@param ["Prodigy", "AdamW8bit", "AdamW", "Adafactor"]
LEARNING_RATE_PRESET = "1.0" #@param ["1.0", "1e-4", "5e-5", "2e-4", "5e-4", "Custom"]
CUSTOM_LEARNING_RATE = 0.0001 #@param {type:"number"}

EPOCHS = 12 #@param {type:"integer"}
REPEATS = 10 #@param {type:"integer"}
MAX_TRAIN_STEPS = 0 #@param {type:"integer"}
NETWORK_DIM_RANK = 32 #@param [16, 32, 64, 128]
NETWORK_ALPHA = 16 #@param [8, 16, 32, 64]
RESOLUTION = 1024 #@param [512, 768, 1024, 1536]
LORA_TYPE = "Standard LoRA" #@param ["Standard LoRA", "LoHa", "LoCon", "DoRA"]

# Giám sát & Thông báo
DISCORD_WEBHOOK_URL = "" #@param {type:"string"}
SAMPLE_PROMPT = "a detailed portrait photo of sks person in dramatic cinematic lighting" #@param {type:"string"}

from lora_colab.core.config import LoRAConfig, DatasetConfig, NetworkConfig, TrainingConfig, ConfigManager
from lora_colab.engines.factory import EngineFactory

# Xử lý Learning Rate
if LEARNING_RATE_PRESET == "1.0":
    resolved_lr = 1.0
elif LEARNING_RATE_PRESET == "1e-4":
    resolved_lr = 1e-4
elif LEARNING_RATE_PRESET == "5e-5":
    resolved_lr = 5e-5
elif LEARNING_RATE_PRESET == "2e-4":
    resolved_lr = 2e-4
elif LEARNING_RATE_PRESET == "5e-4":
    resolved_lr = 5e-4
else:
    resolved_lr = float(CUSTOM_LEARNING_RATE)

# Xử lý Optimizer
if OPTIMIZER_CHOICE == "Prodigy":
    opt_type = "Prodigy"
    resolved_lr = 1.0  # Bắt buộc cho Prodigy D-adaptation
elif OPTIMIZER_CHOICE == "AdamW8bit":
    opt_type = "AdamW8bit"
elif OPTIMIZER_CHOICE == "Adafactor":
    opt_type = "Adafactor"
else:
    opt_type = "AdamW"

active_config = LoRAConfig(
    dataset=DatasetConfig(
        dataset_dir=target_dataset_dir,
        repeats=REPEATS,
        resolution=RESOLUTION,
        enable_bucketing=True
    ),
    network=NetworkConfig(
        network_module="networks.lora" if LORA_TYPE == "Standard LoRA" else "lycoris.kohya",
        network_dim=NETWORK_DIM_RANK,
        network_alpha=NETWORK_ALPHA
    ),
    training=TrainingConfig(
        base_model_path=model_file_path,
        model_family=SELECTED_MODEL,
        output_name=OUTPUT_LORA_NAME,
        output_dir=DRIVE_PATHS["outputs_final_loras"],
        checkpoint_dir=DRIVE_PATHS["outputs_checkpoints"],
        sample_dir=DRIVE_PATHS["outputs_samples"],
        logging_dir=DRIVE_PATHS["outputs_logs"],
        epochs=EPOCHS,
        max_train_steps=MAX_TRAIN_STEPS if MAX_TRAIN_STEPS > 0 else None,
        batch_size=BATCH_SIZE,
        learning_rate=resolved_lr,
        optimizer_type=opt_type,
        discord_webhook_url=DISCORD_WEBHOOK_URL.strip() if DISCORD_WEBHOOK_URL.strip() else None,
        sample_prompt=SAMPLE_PROMPT
    )
)

# Lưu cấu hình hoàn chỉnh vào Google Drive
saved_cfg_path = os.path.join(DRIVE_PATHS["configs"], f"{OUTPUT_LORA_NAME}_config.yaml")
ConfigManager.save_yaml(active_config, saved_cfg_path)
print(f"\n✅ Cell 5: Cấu hình ({opt_type} | LR: {resolved_lr} | Batch: {BATCH_SIZE} | Epochs: {EPOCHS}) đã lưu tại Google Drive:")
print(f"📄 {saved_cfg_path}")

In [ ]:
#@title 🎯 Cell 6: Bắt Đầu Huấn Luyện (Training & Auto-Resume Engine) { vertical-output: true }
#@markdown Khởi chạy quá trình train. Tự động kiểm tra Google Drive để khôi phục (Resume) nếu phiên trước bị ngắt kết nối.

ENABLE_AUTO_RESUME = True #@param {type:"boolean"}

from lora_colab.storage.resume_manager import ResumeManager
from lora_colab.engines.factory import EngineFactory

resume_path = None
if ENABLE_AUTO_RESUME:
    resume_info = ResumeManager.get_resume_status(active_config.training.checkpoint_dir)
    if resume_info["can_resume"]:
        resume_path = resume_info["checkpoint_path"]

trainer = EngineFactory.create_trainer(active_config, engine_choice=TRAINING_ENGINE)
print(f"🚀 Bắt đầu huấn luyện LoRA cho: {active_config.training.model_family}...")
success = trainer.train(resume_from=resume_path)

if success:
    print("\n🎉 Huấn luyện hoàn tất! Các file LoRA đã được lưu an toàn trực tiếp trong Google Drive.")
else:
    print("\n⚠️ Quá trình huấn luyện đã dừng hoặc gặp lỗi.")

In [ ]:
#@title 📦 Cell 7: Kiểm Thử, Xuất File & Upload (Inference, Merge & Export) { vertical-output: true }
#@markdown Test sinh ảnh thử nghiệm với LoRA vừa train, gộp LoRA vào base checkpoint hoặc upload trực tiếp lên HuggingFace.

ACTION = "Test Inference Preview" #@param ["Test Inference Preview", "Merge LoRA into Base Model", "Upload to HuggingFace Hub"]
TEST_PROMPT = "a portrait photo of sks person in beautiful morning lighting, 8k" #@param {type:"string"}
HF_UPLOAD_REPO = "your-username/my-flux-lora" #@param {type:"string"}
HF_TOKEN = "" #@param {type:"string"}

from lora_colab.monitoring.sample_generator import SamplePreviewGenerator
from lora_colab.export.merger import LoRAMerger
from lora_colab.export.uploader import ModelUploader

latest_lora = os.path.join(active_config.training.output_dir, f"{active_config.training.output_name}.safetensors")

if ACTION == "Test Inference Preview":
    preview_out = os.path.join(DRIVE_PATHS["outputs_samples"], "final_preview.png")
    SamplePreviewGenerator.generate_preview(
        base_model_path=model_file_path,
        lora_weights_path=latest_lora if os.path.exists(latest_lora) else None,
        prompt=TEST_PROMPT,
        output_path=preview_out
    )
    from IPython.display import Image as IPImage, display
    if os.path.exists(preview_out):
        display(IPImage(preview_out))

elif ACTION == "Merge LoRA into Base Model":
    merged_dest = os.path.join(DRIVE_PATHS["outputs_final_loras"], f"{active_config.training.output_name}_merged")
    LoRAMerger.merge_lora_to_base(model_file_path, latest_lora, merged_dest)

elif ACTION == "Upload to HuggingFace Hub":
    if os.path.exists(latest_lora):
        ModelUploader.upload_to_huggingface(latest_lora, repo_id=HF_UPLOAD_REPO, token=HF_TOKEN)
    else:
        print(f"File LoRA chưa tồn tại tại: {latest_lora}")

print("\n✅ Cell 7 hoàn tất!")